|                |   |
:----------------|---|
| **Nombre**     |Santiago Escutia Ríos   |
| **Fecha**      | 19/2/2026  |
| **Expediente** |757839   |

Carga de datos y se quita la columna model porque no es relevante.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [14]:
df= pd.read_excel("Motor Trend Car Road Tests.xlsx")
df_num = df.drop(columns=['model'])

Aquí saqué los R² y los betas. Por los signos se ve que el peso (wt) y la potencia (hp) afectan negativamente el mpg, mientras más pesado o más potente el carro, más gasolina consume.

Usé 40% para entrenar y por eso el R² de prueba salió bajo. Para mejorarlo apliqué Ridge (L2); con λ = 10 el modelo se vuelve más estable con datos nuevos y el R² de prueba aumenta.

Se ajusta una una regresión lineal para predecir el consumo (mpg) usando el resto de las variables (menos model). El objetivo es ver qué influye en la eficiencia y qué tan exacto es el modelo con el R².

In [23]:
def run_analysis_numeric(target_col):
    X = df_num.drop(columns=[target_col])
    y = df_num[target_col]
    
    model_full = LinearRegression()
    model_full.fit(X, y)
    r2_full = model_full.score(X, y)
    
    print(f"Análisis para {target_col} (Numérico)")
    print(f"R2 Modelo Completo: {r2_full:.4f}")
    print("Betas (Signos):")
    print(pd.Series(model_full.coef_, index=X.columns))
    
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.4, random_state=42)
    
    model_split = LinearRegression()
    model_split.fit(X_train, y_train)
    print(f"R2 Entrenamiento: {model_split.score(X_train, y_train):.4f}")
    print(f"R2 Prueba: {model_split.score(X_test, y_test):.4f}")
    
    #Regularización L2 (Ridge)
    print("\nComparación Regularización L2 (Alpha = Lambda):")
    for a in [0.1, 1.0, 10.0, 100.0]:
        ridge = Ridge(alpha=a)
        ridge.fit(X_train, y_train)
        print(f"Lambda {a}: R2 Train {ridge.score(X_train, y_train):.3f} | R2 Test {ridge.score(X_test, y_test):.3f}")
    print("-" * 40)
    return r2_full


El R² de entrenamiento es mucho más alto que el de prueba, lo que significa que el modelo se memorizó los datos en lugar de aprender. Está sobreajustado porque el dataset es muy chico.

In [16]:
r2_1_1 = run_analysis_numeric('mpg')
r2_1_2 = run_analysis_numeric('qsec')

--- Análisis para mpg (Numérico) ---
R2 Modelo Completo: 0.8690
Betas (Signos):
cyl    -0.111440
disp    0.013335
hp     -0.021482
drat    0.787111
wt     -3.715304
qsec    0.821041
vs      0.317763
am      2.520227
gear    0.655413
carb   -0.199419
dtype: float64
R2 Entrenamiento: 0.9982
R2 Prueba: -7.1071

Comparación Regularización L2 (Alpha = Lambda):
Lambda 0.1: R2 Train 0.979 | R2 Test 0.261
Lambda 1.0: R2 Train 0.928 | R2 Test 0.631
Lambda 10.0: R2 Train 0.863 | R2 Test 0.657
Lambda 100.0: R2 Train 0.807 | R2 Test 0.599
----------------------------------------
--- Análisis para qsec (Numérico) ---
R2 Modelo Completo: 0.8747
Betas (Signos):
mpg     0.069048
cyl    -0.362678
disp   -0.007501
hp     -0.001563
drat   -0.131064
wt      1.496332
vs      0.970035
am     -0.901186
gear   -0.201285
carb   -0.273598
dtype: float64
R2 Entrenamiento: 0.9989
R2 Prueba: -1.0013

Comparación Regularización L2 (Alpha = Lambda):
Lambda 0.1: R2 Train 0.986 | R2 Test 0.688
Lambda 1.0: R2 Train 0.9

La regularización para el sobreajuste, al ajustar el valor de λ, el modelo deja de memorizar y empieza a funcionar bien con datos nuevos.

El peso y la potencia bajaron el rendimiento, mientras que la transmisión manual lo mejoró. Estos resultados tienen todo el sentido lógicamente.

Se convirtió cyl, gear y carb a dummies para probar.

El R² de entrenamiento quedó casi en 1, pero el de prueba bajó mucho. Con pocos datos y más variables, el modelo se sobreajustó y memorizó en vez de predecir bien.

In [22]:
#Dummies
df_dummies = pd.get_dummies(df_num, columns=['cyl', 'gear', 'carb'], drop_first=True)

def run_analysis_dummies(target_col):
    X = df_dummies.drop(columns=[target_col])
    y = df_dummies[target_col]
    
    #Modelo 
    model_full = LinearRegression()
    model_full.fit(X, y)
    r2_full = model_full.score(X, y)
    
    # Split 40% entrenamiento
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.4, random_state=42)
    model_split = LinearRegression()
    model_split.fit(X_train, y_train)
    
    print(f"Análisis para {target_col} (Dummies)")
    print(f"R2 modelo completo: {r2_full:.4f}")
    print(f"R2 entrenamiento: {model_split.score(X_train, y_train):.4f}")
    print(f"R2 prueba: {model_split.score(X_test, y_test):.4f}")
    print("-" * 40)
    return r2_full

r2_2_1 = run_analysis_dummies('mpg')
r2_2_2 = run_analysis_dummies('qsec')

Análisis para mpg (Dummies)
R2 modelo completo: 0.8931
R2 entrenamiento: 1.0000
R2 prueba: -1.3253
----------------------------------------
Análisis para qsec (Dummies)
R2 modelo completo: 0.9083
R2 entrenamiento: 1.0000
R2 prueba: -0.0600
----------------------------------------


Los dummies ayudan un poco, pero sin regularización el modelo se sigue sobreajustando en ambos casos.

El usar variables dummy sube un poco el R² porque ayuda a distinguir mejor las categorías, así el modelo entiende mejor los datos.

Los modelos numéricos del punto 1 son mejores. Las dummies dan un R² alto, pero es overfitting y no sirven para predecir bien, me quedo con el numérico usando Ridge para más estabilidad.